<a href="https://colab.research.google.com/github/adminsanjay/ML-projects/blob/main/fake_audio_detecter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import os
import kagglehub

# Set the download directory to sample_data
os.environ["KAGGLEHUB_CACHE"] = "./sample_data"

# Download latest version
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
path = "./sample_data/datasets/awsaf49/asvpoof-2019-dataset/versions/1"
print("Path to dataset files:", path)


100%|██████████| 23.6G/23.6G [04:36<00:00, 91.4MB/s]

Extracting files...


Path to dataset files: ./sample_data/datasets/awsaf49/asvpoof-2019-dataset/versions/1


In [6]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import scipy.fftpack as fftpack
import soundfile as sf


In [15]:
#Configuration
DATASET_PATH = "./sample_data/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA"
TRAIN_DIR = os.path.join(DATASET_PATH, "ASVspoof2019_LA_train/flac")
PROTOCOL_FILE = os.path.join(DATASET_PATH, "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")

SR = 16000 # Standard for ASVspoof
DURATION = 3
SAMPLES = SR * DURATION

In [16]:
def extract_lfcc(file_path, n_lfcc=40, n_filters=40):
    try:
        # 1. Use soundfile for FLAC (More stable than librosa.load for this format)
        audio, native_sr = sf.read(file_path)

        # Resample if necessary
        if native_sr != SR:
            audio = librosa.resample(audio, orig_sr=native_sr, target_sr=SR)

        # 2. Trim/Pad
        if len(audio) > SAMPLES:
            audio = audio[:SAMPLES]
        else:
            audio = np.pad(audio, (0, SAMPLES - len(audio)), mode='constant')

        # 3. Compute Linear Spectrogram
        # Use a power spectrogram for better artifact detection
        S = np.abs(librosa.stft(audio, n_fft=2048))**2

        # 4. Create Linear Filter Bank manually (Librosa's mel filters can be adapted)
        # This is the "Linear" part that captures high-freq AI artifacts
        linear_basis = librosa.filters.mel(sr=SR, n_fft=2048, n_mels=n_filters, fmin=0, fmax=SR/2, htk=True)
        # Note: Setting htk=True makes the filter spacing more linear across the range

        features = np.dot(linear_basis, S)
        lfcc = librosa.feature.mfcc(S=librosa.power_to_db(features), n_mfcc=n_lfcc)

        # 5. Normalization
        lfcc = (lfcc - np.mean(lfcc)) / (np.std(lfcc) + 1e-9)
        return lfcc

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [17]:
def apply_spec_augment(spec, num_mask=2, freq_mask_width=5, time_mask_width=10):
    #Applies Frequency and Time masking to LFCC features (SpecAugment).
    # Create a copy so we don't modify the original data
    augmented_spec = spec.copy()
    n_freqs, n_frames = augmented_spec.shape

    # 1. Frequency Masking (Horizontal strips in LFCC)
    for i in range(num_mask):
        f = np.random.randint(0, freq_mask_width)
        f0 = np.random.randint(0, n_freqs - f)
        augmented_spec[f0:f0 + f, :] = np.mean(augmented_spec)

    # 2. Time Masking (Vertical strips in LFCC)
    for i in range(num_mask):
        t = np.random.randint(0, time_mask_width)
        t0 = np.random.randint(0, n_frames - t)
        augmented_spec[:, t0:t0 + t] = np.mean(augmented_spec)

    return augmented_spec

In [18]:
df = pd.read_csv(PROTOCOL_FILE, sep=" ", header=None, names=['speaker', 'filename', 'v3', 'v4', 'label'])

X = []
Y = []

print("Extracting LFCC features...")
for index, row in df.head(2000).iterrows():
    file_path = os.path.join(TRAIN_DIR, f"{row['filename']}.flac")
    features = extract_lfcc(file_path) # Using the new LFCC function

    if features is not None:
        label = 1 if row['label'] == 'bonafide' else 0
        X.append(features)
        Y.append(label)
        # Add Augmented Version (SpecAugment)
        aug_features = apply_spec_augment(features)
        X.append(aug_features)
        Y.append(label)

# Convert to Numpy Arrays
X = np.array(X)
y = np.array(Y)

# Split into Training (80%) and Testing (20%)
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"Data Loaded! Training shape: {x_train.shape}, Labels: {len(y_train)}")

# Add the channel dimension
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print(f"Final Input Shape for ResNet: {x_train.shape[:]}")

Extracting LFCC features...
Data Loaded! Training shape: (3200, 40, 94), Labels: 3200
Final Input Shape for ResNet: (3200, 40, 94, 1)
